# Catalog Reference Finder
Lists every object that lives in, or refers to, a Unity Catalog catalog, so you can plan a rename.

| Category | Meaning |
|---|---|
| 1 - Inside catalog | Moves automatically with `ALTER CATALOG ... RENAME TO` |
| 2 - References catalog name | Contains the catalog name as text, so it **must be updated** |
| 3 - Usage evidence | Read or wrote the catalog recently (lineage, query history, shares), so **verify** it |

Run this as a **workspace admin** with SELECT on the `system.access`, `system.query` and `system.information_schema` tables for complete results.
Non-admins only see the objects they have permission on.

In [0]:
%pip install --upgrade databricks-sdk --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog_name", "logistics", "Catalog name")
dbutils.widgets.text("workspace_root", "/", "Workspace folder to scan")
dbutils.widgets.dropdown("scan_workspace_files", "yes", ["yes", "no"], "Scan notebooks & files")
dbutils.widgets.text("lookback_days", "90", "Lineage / query history days")
dbutils.widgets.text("output_table", "", "Save results to table (optional)")

CATALOG = dbutils.widgets.get("catalog_name").strip()
ROOT = dbutils.widgets.get("workspace_root").strip() or "/"
SCAN_FILES = dbutils.widgets.get("scan_workspace_files") == "yes"
DAYS = int(dbutils.widgets.get("lookback_days"))
OUTPUT_TABLE = dbutils.widgets.get("output_table").strip()
print(f"Catalog: {CATALOG} | workspace root: {ROOT} | lookback: {DAYS} days")

Catalog: logistics | workspace root: / | lookback: 90 days


In [0]:
import base64, json, re
from concurrent.futures import ThreadPoolExecutor
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ObjectType

w = WorkspaceClient()
PATTERN = re.compile(re.escape(CATALOG), re.IGNORECASE)

IN_CATALOG = "1 - Inside catalog (moves with rename)"
REFERENCES = "2 - References catalog name (must update)"
USAGE = "3 - Usage evidence (verify)"

results, errors = [], []
JOB_NAMES, PIPELINE_NAMES = {}, {}


def add(category, object_type, object_id, name, detail="", owner=""):
    results.append({
        "category": category,
        "object_type": str(object_type or ""),
        "object_id": str(object_id or ""),
        "name": str(name or ""),
        "detail": str(detail or "")[:500],
        "owner": str(owner or ""),
    })


def section(title):
    """Run a section; one failing section (e.g. missing permission) doesn't stop the rest."""
    def deco(fn):
        before = len(results)
        try:
            fn()
            print(f"[ok]   {title}: {len(results) - before} found")
        except Exception as e:
            errors.append((title, repr(e)[:500]))
            print(f"[skip] {title}: {e}")
        return fn
    return deco


def snippet(text):
    for i, line in enumerate(text.splitlines(), 1):
        if PATTERN.search(line):
            return f"line {i}: {line.strip()[:200]}"
    return ""


def q(sql, **args):
    return spark.sql(sql, args=args).collect()

## 1. Objects inside the catalog

In [0]:
@section("Schemas")
def _():
    for r in q("""SELECT schema_name, schema_owner FROM system.information_schema.schemata
                  WHERE catalog_name = :cat AND schema_name <> 'information_schema'""", cat=CATALOG):
        add(IN_CATALOG, "SCHEMA", f"{CATALOG}.{r.schema_name}", r.schema_name, owner=r.schema_owner)


@section("Tables / views / materialized views / streaming tables")
def _():
    for r in q("""SELECT table_schema, table_name, table_type, table_owner FROM system.information_schema.tables
                  WHERE table_catalog = :cat AND table_schema <> 'information_schema'""", cat=CATALOG):
        detail = "review view SQL after rename" if "VIEW" in (r.table_type or "") else ""
        add(IN_CATALOG, r.table_type, f"{CATALOG}.{r.table_schema}.{r.table_name}", r.table_name, detail, r.table_owner)


@section("Functions")
def _():
    for r in q("""SELECT routine_schema, routine_name, routine_type FROM system.information_schema.routines
                  WHERE routine_catalog = :cat""", cat=CATALOG):
        add(IN_CATALOG, f"FUNCTION ({r.routine_type})", f"{CATALOG}.{r.routine_schema}.{r.routine_name}", r.routine_name)


@section("Volumes")
def _():
    for r in q("""SELECT volume_schema, volume_name, volume_type, volume_owner FROM system.information_schema.volumes
                  WHERE volume_catalog = :cat""", cat=CATALOG):
        add(IN_CATALOG, f"VOLUME ({r.volume_type})", f"{CATALOG}.{r.volume_schema}.{r.volume_name}", r.volume_name,
            owner=r.volume_owner)


@section("Registered models")
def _():
    schemas = q("""SELECT schema_name FROM system.information_schema.schemata
                   WHERE catalog_name = :cat AND schema_name <> 'information_schema'""", cat=CATALOG)
    for s in schemas:
        for m in w.registered_models.list(catalog_name=CATALOG, schema_name=s.schema_name):
            add(IN_CATALOG, "MODEL", m.full_name, m.name, owner=m.owner)

[ok]   Schemas: 4 found
[ok]   Tables / views / materialized views / streaming tables: 21 found
[ok]   Functions: 0 found
[ok]   Volumes: 0 found
[ok]   Registered models: 0 found


## 2. Objects that reference the catalog name as text (must update)

In [0]:
@section("View definitions (all catalogs)")
def _():
    for r in q("""SELECT table_catalog, table_schema, table_name, view_definition FROM system.information_schema.views
                  WHERE instr(lower(view_definition), lower(:cat)) > 0""", cat=CATALOG):
        add(REFERENCES, "VIEW", f"{r.table_catalog}.{r.table_schema}.{r.table_name}", r.table_name,
            snippet(r.view_definition or ""))


@section("Function bodies (all catalogs)")
def _():
    for r in q("""SELECT routine_catalog, routine_schema, routine_name, routine_definition
                  FROM system.information_schema.routines
                  WHERE instr(lower(routine_definition), lower(:cat)) > 0""", cat=CATALOG):
        add(REFERENCES, "FUNCTION", f"{r.routine_catalog}.{r.routine_schema}.{r.routine_name}", r.routine_name,
            snippet(r.routine_definition or ""))


@section("Jobs (settings, parameters, task configs)")
def _():
    for j in w.jobs.list(expand_tasks=True):
        settings = j.settings.as_dict() if j.settings else {}
        JOB_NAMES[str(j.job_id)] = settings.get("name", "")
        text = json.dumps(settings)
        if PATTERN.search(text):
            add(REFERENCES, "JOB", j.job_id, settings.get("name"),
                f"{len(PATTERN.findall(text))} match(es) in job settings", j.creator_user_name)


@section("Pipelines (DLT / Lakeflow)")
def _():
    for p in w.pipelines.list_pipelines():
        PIPELINE_NAMES[p.pipeline_id] = p.name
        d = w.pipelines.get(p.pipeline_id)
        spec = d.spec.as_dict() if d.spec else {}
        if PATTERN.search(json.dumps(spec)):
            add(REFERENCES, "PIPELINE", p.pipeline_id, p.name,
                f"target catalog={spec.get('catalog')}, schema={spec.get('schema') or spec.get('target')}",
                d.creator_user_name)


matched_query_ids = set()


@section("SQL queries")
def _():
    try:
        items = [(x.id, x.display_name, x.query_text, x.owner_user_name) for x in w.queries.list()]
    except Exception:  # older workspaces / SDKs
        items = [(x.id, x.name, x.query, getattr(x.user, "email", "")) for x in w.queries_legacy.list()]
    for qid, name, text, owner in items:
        if text and PATTERN.search(text):
            matched_query_ids.add(qid)
            add(REFERENCES, "SQL_QUERY", qid, name, snippet(text), owner)


@section("SQL alerts (built on matching queries)")
def _():
    for a in w.alerts.list():
        if a.query_id in matched_query_ids:
            add(REFERENCES, "ALERT", a.id, a.display_name, f"uses query {a.query_id}", a.owner_user_name)


@section("AI/BI (Lakeview) dashboards")
def _():
    for d in w.lakeview.list():
        full = w.lakeview.get(d.dashboard_id)
        if PATTERN.search(full.serialized_dashboard or ""):
            add(REFERENCES, "DASHBOARD", d.dashboard_id, d.display_name, d.path)


TEXT_EXT = (".py", ".sql", ".scala", ".r", ".yml", ".yaml", ".json", ".txt", ".sh", ".tf", ".cfg", ".ini", ".toml")

if SCAN_FILES:
    @section(f"Notebooks & workspace files under {ROOT}")
    def _():
        objs = [o for o in w.workspace.list(ROOT, recursive=True)
                if o.object_type == ObjectType.NOTEBOOK
                or (o.object_type == ObjectType.FILE and o.path.lower().endswith(TEXT_EXT))]
        print(f"       scanning {len(objs)} notebooks/files ...")

        def scan(o):
            try:
                fmt = ExportFormat.SOURCE if o.object_type == ObjectType.NOTEBOOK else ExportFormat.AUTO
                content = w.workspace.export(o.path, format=fmt).content
                text = base64.b64decode(content).decode("utf-8", errors="ignore") if content else ""
                if PATTERN.search(text):
                    return o, len(PATTERN.findall(text)), snippet(text)
            except Exception as e:
                errors.append((o.path, repr(e)[:300]))
            return None

        with ThreadPoolExecutor(max_workers=16) as ex:
            for hit in ex.map(scan, objs):
                if hit:
                    o, n, snip = hit
                    add(REFERENCES, o.object_type.value, o.object_id, o.path, f"{n} match(es); {snip}")

[ok]   View definitions (all catalogs): 32 found
[ok]   Function bodies (all catalogs): 0 found
[ok]   Jobs (settings, parameters, task configs): 0 found
[ok]   Pipelines (DLT / Lakeflow): 0 found
[ok]   SQL queries: 0 found
[ok]   SQL alerts (built on matching queries): 0 found
[ok]   AI/BI (Lakeview) dashboards: 0 found
       scanning 48 notebooks/files ...
[ok]   Notebooks & workspace files under /: 4 found


## 3. Usage evidence (who actually reads/writes the catalog)

In [0]:
@section("Lineage: jobs, notebooks, pipelines, dashboards, queries")
def _():
    rows = q("""
        SELECT entity_type, entity_id,
               max(event_time) AS last_seen,
               concat_ws(', ', collect_set(created_by)) AS users,
               concat_ws(', ', slice(collect_set(coalesce(source_table_full_name, target_table_full_name)), 1, 5)) AS sample_tables
        FROM system.access.table_lineage
        WHERE event_date >= date_sub(current_date(), :days)
          AND (source_table_catalog = :cat OR target_table_catalog = :cat)
          AND entity_type IS NOT NULL
        GROUP BY entity_type, entity_id""", cat=CATALOG, days=DAYS)
    for r in rows:
        name = JOB_NAMES.get(str(r.entity_id)) or PIPELINE_NAMES.get(str(r.entity_id)) or ""
        add(USAGE, f"{r.entity_type} (lineage)", r.entity_id, name,
            f"last seen {r.last_seen}; tables: {r.sample_tables}", r.users)


@section("Query history: SQL warehouses, BI tools, external clients")
def _():
    rows = q("""
        SELECT client_application, executed_by, count(*) AS runs, max(start_time) AS last_run,
               first(statement_text) AS sample
        FROM system.query.history
        WHERE start_time >= date_sub(current_date(), :days)
          AND instr(lower(statement_text), lower(:cat)) > 0
        GROUP BY client_application, executed_by""", cat=CATALOG, days=DAYS)
    for r in rows:
        add(USAGE, "SQL CLIENT", r.client_application, r.executed_by,
            f"{r.runs} runs, last {r.last_run}; e.g. {(r.sample or '')[:200]}", r.executed_by)


@section("Delta Sharing shares")
def _():
    for s in w.shares.list_shares():
        full = w.shares.get(s.name, include_shared_data=True)
        hits = [o.name for o in (full.objects or []) if o.name and o.name.lower().startswith(CATALOG.lower() + ".")]
        if hits:
            add(USAGE, "SHARE", s.name, s.name, f"{len(hits)} shared object(s): {', '.join(hits[:5])}", s.owner)

[ok]   Lineage: jobs, notebooks, pipelines, dashboards, queries: 4 found
[ok]   Query history: SQL warehouses, BI tools, external clients: 1 found
[ok]   Delta Sharing shares: 0 found


## Results

In [0]:
COLS = ["category", "object_type", "object_id", "name", "detail", "owner"]
df = spark.createDataFrame([tuple(r[c] for c in COLS) for r in results],
                           schema=", ".join(f"{c} string" for c in COLS))

print(f"Total objects found: {df.count()}")
display(df.groupBy("category", "object_type").count().orderBy("category", "object_type"))

Total objects found: 66


category,object_type,count
1 - Inside catalog (moves with rename),MANAGED,20
1 - Inside catalog (moves with rename),SCHEMA,4
1 - Inside catalog (moves with rename),VIEW,1
2 - References catalog name (must update),NOTEBOOK,4
2 - References catalog name (must update),VIEW,32
3 - Usage evidence (verify),NOTEBOOK (lineage),4
3 - Usage evidence (verify),SQL CLIENT,1


In [0]:
display(df.orderBy("category", "object_type", "name"))

category,object_type,object_id,name,detail,owner
1 - Inside catalog (moves with rename),MANAGED,logistics.bronze.carriers,carriers,,agenticaibatch5@gmail.com
1 - Inside catalog (moves with rename),MANAGED,logistics.silver.carriers_silver,carriers_silver,,agenticaibatch5@gmail.com
1 - Inside catalog (moves with rename),MANAGED,logistics.bronze.delivery_events,delivery_events,,agenticaibatch5@gmail.com
1 - Inside catalog (moves with rename),MANAGED,logistics.silver.delivery_events_silver,delivery_events_silver,,agenticaibatch5@gmail.com
1 - Inside catalog (moves with rename),MANAGED,logistics.gold.dim_carrier,dim_carrier,,agenticaibatch5@gmail.com
1 - Inside catalog (moves with rename),MANAGED,logistics.gold.dim_date,dim_date,,agenticaibatch5@gmail.com
1 - Inside catalog (moves with rename),MANAGED,logistics.gold.dim_route,dim_route,,agenticaibatch5@gmail.com
1 - Inside catalog (moves with rename),MANAGED,logistics.gold.dim_warehouse,dim_warehouse,,agenticaibatch5@gmail.com
1 - Inside catalog (moves with rename),MANAGED,logistics.gold.fact_delivery_events,fact_delivery_events,,agenticaibatch5@gmail.com
1 - Inside catalog (moves with rename),MANAGED,logistics.gold.fact_shipments,fact_shipments,,agenticaibatch5@gmail.com


In [0]:
if OUTPUT_TABLE:
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE)
    print(f"Saved to {OUTPUT_TABLE}")

if errors:
    print(f"{len(errors)} section(s)/object(s) could not be scanned (usually permissions):")
    display(spark.createDataFrame(errors, schema="source string, error string"))